# Experiment: Case-001 zero-country non-CTG source radar

Objective: validate the July 12 non-ClinicalTrials.gov source map for countries that returned zero active-status ClinicalTrials.gov records on July 11.

Success criteria: every zero-country label has at least one non-ClinicalTrials.gov source route, all source routes remain owner-verification-only, and no patient-action output is emitted.

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd()
RADAR_PATH = (
    ROOT / "data/workflows/case-001/2026-07-12-zero-country-non-ctg-source-radar.json"
)
payload = json.loads(RADAR_PATH.read_text(encoding="utf-8"))
payload["current_label"]

## Plan

- Check that the July 11 zero-country labels are preserved.
- Check that non-ClinicalTrials.gov source routes cover each zero-country label.
- Check that source routes are not promoted to patient-specific action.
- Check that the Indonesia gene-therapy case-report signal is kept as owner-verification context only.

In [ ]:
expected_zero_countries = {"Hong Kong", "Indonesia", "Singapore", "Iran", "Russia"}
zero_countries = set(payload["prior_input"]["zero_country_labels"])

assert payload["checked_date"] == "2026-07-12"
assert zero_countries == expected_zero_countries
assert payload["case001_state"] == "branch_review_handoff_packet_incomplete"

routes_by_country: dict[str, list[dict[str, object]]] = defaultdict(list)
for route in payload["source_routes"]:
    country = str(route["country"])
    routes_by_country[country].append(route)

assert set(routes_by_country) == expected_zero_countries
assert all(routes_by_country[country] for country in expected_zero_countries)
Counter(route["country"] for route in payload["source_routes"])

In [ ]:
route_ids = {route["route_id"] for route in payload["source_routes"]}
required_routes = {
    "INA_REGISTRY_CENTER",
    "INDONESIA_GENE_THERAPY_CASE_REPORT_2026",
    "HSA_CLINICAL_TRIALS_REGISTER",
    "HKUCTR",
    "IRCT20090613002027N16",
    "WHO_OTHER_REGISTRY_RUSSIAN_STATE_REGISTER",
}

assert required_routes.issubset(route_ids)
assert all(route["patient_action_claim"] is False for route in payload["source_routes"])

indonesia_routes = routes_by_country["Indonesia"]
assert any(
    route["route_id"] == "INDONESIA_GENE_THERAPY_CASE_REPORT_2026"
    for route in indonesia_routes
)
assert any(
    route["route_id"] == "BPOM_ATMP_CURRENT_SECONDARY_SIGNAL"
    for route in indonesia_routes
)

iran_trial_ids = [
    route["route_id"]
    for route in routes_by_country["Iran"]
    if route["route_id"].startswith("IRCT")
]
assert len(iran_trial_ids) == 2
iran_trial_ids

In [ ]:
pubmed_pmids = {
    pmid for pmids in payload["supporting_pmids"].values() for pmid in pmids
}

assert {"35950580", "41883840", "39911648", "31272240"}.issubset(pubmed_pmids)

allowed = set(payload["allowed_outputs"])
blocked = set(payload["blocked_outputs"])
assert allowed.isdisjoint(blocked)
assert "treatment_selection" in blocked
assert "case_report_not_access_proof" in allowed

summary = {
    "label": payload["current_label"],
    "zero_country_count": len(zero_countries),
    "source_route_count": len(payload["source_routes"]),
    "countries": sorted(routes_by_country),
    "patient_action_state": "blocked",
}
summary

## Results

- All five July 11 zero-country labels now have non-ClinicalTrials.gov source routes to verify.
- The Indonesia 2026 gene-therapy case report is the most concrete new public source signal, but it is a single-case owner-verification lead and not proof of local availability, eligibility, referral, travel, import, purchase, or treatment selection.
- Iran has direct IRCT thalassemia records in this source map, but they are completed supportive/iron-risk records, not curative active-access signals.
- Russia remains a weaker source route because WHO lists the Russian State Register under other registries, not as a standalone ICMJE/Primary Registry route.

## Next steps

- Query the national registry interfaces directly where search forms are usable.
- For the Indonesian case report, ask qualified hematology/regulatory owners what it means for access pathway, manufacturing route, payer route, and follow-up infrastructure.
- Keep Case-001 branch selection blocked until private packet readiness and clinician review change.